#Upload and Read Excel file

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from scipy.stats import shapiro, ttest_rel

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print("PRS Excel file loaded successfully.")
display(df.head())
print(df.columns)

Saving PRS_fake_2_participants.xlsx to PRS_fake_2_participants.xlsx
PRS Excel file loaded successfully.


,ParticipantID,Condition,BA1,BA2,BA3,FAS4,FAS5,FAS6,COH7,COH8,COH9,SCO10,SCO11
0,P01,Smell,6.0,6.0,5.0,6.0,5.0,6.0,5.0,6.0,5.0,6.0,5.0
1,P01,NoSmell,4.0,4.0,3.0,4.0,3.0,4.0,4.0,3.0,4.0,3.0,4.0
2,P02,Smell,5.0,6.0,5.0,5.0,6.0,5.0,6.0,5.0,5.0,5.0,6.0
3,P02,NoSmell,3.0,4.0,4.0,3.0,4.0,3.0,4.0,4.0,3.0,4.0,3.0
4,P03,Smell,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Index(['ParticipantID', 'Condition', 'BA1', 'BA2', 'BA3', 'FAS4', 'FAS5',
       'FAS6', 'COH7', 'COH8', 'COH9', 'SCO10', 'SCO11'],
      dtype='object')


#Check required columns

In [ ]:
required_columns = [
    "ParticipantID",
    "Condition",
    "BA1", "BA2", "BA3",
    "FAS4", "FAS5", "FAS6",
    "COH7", "COH8", "COH9",
    "SCO10", "SCO11"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:")
    for col in missing_columns:
        print("-", col)
else:
    print("All required PRS columns are present.")

df["Condition"] = df["Condition"].astype(str).str.strip()

condition_mapping = {
    "smell": "Smell",
    "Smell": "Smell",
    "SMELL": "Smell",
    "no_smell": "NoSmell",
    "NoSmell": "NoSmell",
    "No-Smell": "NoSmell",
    "no-smell": "NoSmell",
    "No smell": "NoSmell",
    "no smell": "NoSmell",
    "control": "NoSmell",
    "Control": "NoSmell"
}

df["Condition"] = df["Condition"].replace(condition_mapping)

print(df["Condition"].value_counts())

All required PRS columns are present.
Condition
Smell      12
NoSmell    12
Name: count, dtype: int64


#Calculate PRS primary outcomes

In [ ]:
# Calculate PRS dimension means
df["BeingAway"] = df[["BA1", "BA2", "BA3"]].mean(axis=1)

df["Fascination"] = df[["FAS4", "FAS5", "FAS6"]].mean(axis=1)

df["Coherence"] = df[["COH7", "COH8", "COH9"]].mean(axis=1)

df["Scope"] = df[["SCO10", "SCO11"]].mean(axis=1)

df["TotalPRS"] = df[
    [
        "BA1", "BA2", "BA3",
        "FAS4", "FAS5", "FAS6",
        "COH7", "COH8", "COH9",
        "SCO10", "SCO11"
    ]
].mean(axis=1)

# Keep only participant-level PRS scores
prs_scores = df[
    [
        "ParticipantID",
        "Condition",
        "TotalPRS",
        "BeingAway",
        "Fascination",
        "Coherence",
        "Scope"
    ]
].copy()

# Convert to wide format: one row per participant
wide_df = prs_scores.pivot(
    index="ParticipantID",
    columns="Condition",
    values=[
        "TotalPRS",
        "BeingAway",
        "Fascination",
        "Coherence",
        "Scope"
    ]
)

# Flatten column names
wide_df.columns = [
    f"{score}_{condition}"
    for score, condition in wide_df.columns
]

wide_df = wide_df.reset_index()

# Reorder columns
wide_df = wide_df[
    [
        "ParticipantID",
        "TotalPRS_NoSmell",
        "TotalPRS_Smell",
        "BeingAway_NoSmell",
        "BeingAway_Smell",
        "Fascination_NoSmell",
        "Fascination_Smell",
        "Coherence_NoSmell",
        "Coherence_Smell",
        "Scope_NoSmell",
        "Scope_Smell"
    ]
]

wide_df = wide_df.sort_values("ParticipantID")

print("Final participant-level PRS primary outcomes:")
display(wide_df)

primary_outcomes_file = "PRS_PrimaryOutcomes.xlsx"

wide_df.to_excel(
    primary_outcomes_file,
    index=False
)

print(f"{primary_outcomes_file} saved.")

Final participant-level PRS primary outcomes:


,ParticipantID,TotalPRS_NoSmell,TotalPRS_Smell,BeingAway_NoSmell,BeingAway_Smell,Fascination_NoSmell,Fascination_Smell,Coherence_NoSmell,Coherence_Smell,Scope_NoSmell,Scope_Smell
0,P01,3.636364,5.545455,3.666667,5.666667,3.666667,5.666667,3.666667,5.333333,3.5,5.5
1,P02,3.545455,5.363636,3.666667,5.333333,3.333333,5.333333,3.666667,5.333333,3.5,5.5
2,P03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,P06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,P07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,P08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,P09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,P10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


PRS_PrimaryOutcomes.xlsx saved.


#Descriptive statistics

In [ ]:
descriptive_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    descriptive_results.append({
        "PRS_Measure": measure_name,
        "Condition": "No-Smell",
        "Mean": wide_df[no_smell_col].mean(),
        "SD": wide_df[no_smell_col].std(ddof=1)
    })

    descriptive_results.append({
        "PRS_Measure": measure_name,
        "Condition": "Smell",
        "Mean": wide_df[smell_col].mean(),
        "SD": wide_df[smell_col].std(ddof=1)
    })

descriptive_df = pd.DataFrame(descriptive_results)

display(descriptive_df)

descriptive_file = "PRS_DescriptiveStatistics.xlsx" #Table 5.6

descriptive_df.to_excel(
    descriptive_file,
    index=False
)

print(f"{descriptive_file} saved.")

,PRS_Measure,Condition,Mean,SD
0,Total PRS,No-Smell,3.590909,0.064282
1,Total PRS,Smell,5.454545,0.128565
2,Being Away,No-Smell,3.666667,0.000000
3,Being Away,Smell,5.500000,0.235702
4,Fascination,No-Smell,3.500000,0.235702
5,Fascination,Smell,5.500000,0.235702
6,Coherence,No-Smell,3.666667,0.000000
7,Coherence,Smell,5.333333,0.000000
8,Scope,No-Smell,3.500000,0.000000
9,Scope,Smell,5.500000,0.000000


PRS_DescriptiveStatistics.xlsx saved.


#Shapiro–Wilk normality test

In [ ]:
normality_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    difference_scores = (
        wide_df[smell_col] - wide_df[no_smell_col]
    ).dropna()

    if len(difference_scores) >= 3:
        stat, p_value = shapiro(difference_scores)

        normality_results.append({
            "PRS_Measure": measure_name,
            "Shapiro_W": stat,
            "p_value": p_value,
            "Normality_Assumption": "Met" if p_value >= 0.05 else "Not met"
        })

    else:
        normality_results.append({
            "PRS_Measure": measure_name,
            "Shapiro_W": np.nan,
            "p_value": np.nan,
            "Normality_Assumption": "Not enough data"
        })

normality_df = pd.DataFrame(normality_results)

display(normality_df)

normality_file = "PRS_NormalityResults.xlsx" #Table F.3

normality_df.to_excel(
    normality_file,
    index=False
)

print(f"{normality_file} saved.")

,PRS_Measure,Shapiro_W,p_value,Normality_Assumption
0,Total PRS,NaN,NaN,Not enough data
1,Being Away,NaN,NaN,Not enough data
2,Fascination,NaN,NaN,Not enough data
3,Coherence,NaN,NaN,Not enough data
4,Scope,NaN,NaN,Not enough data


PRS_NormalityResults.xlsx saved.


#Paired-samples t-test

In [ ]:
ttest_results = []

for measure_name, no_smell_col, smell_col in [
    ("Total PRS", "TotalPRS_NoSmell", "TotalPRS_Smell"),
    ("Being Away", "BeingAway_NoSmell", "BeingAway_Smell"),
    ("Fascination", "Fascination_NoSmell", "Fascination_Smell"),
    ("Coherence", "Coherence_NoSmell", "Coherence_Smell"),
    ("Scope", "Scope_NoSmell", "Scope_Smell")
]:

    paired_data = wide_df[
        ["ParticipantID", no_smell_col, smell_col]
    ].dropna()

    no_smell_values = paired_data[no_smell_col]
    smell_values = paired_data[smell_col]

    difference_scores = smell_values - no_smell_values

    if len(paired_data) >= 2:
        t_stat, p_value = ttest_rel(
            smell_values,
            no_smell_values
        )

        difference_sd = difference_scores.std(ddof=1)

        cohens_d = (
            difference_scores.mean() / difference_sd
            if difference_sd != 0 else np.nan
        )

        ttest_results.append({
            "PRS_Measure": measure_name,
            "NoSmell_Mean": no_smell_values.mean(),
            "NoSmell_SD": no_smell_values.std(ddof=1),
            "Smell_Mean": smell_values.mean(),
            "Smell_SD": smell_values.std(ddof=1),
            "Mean_Difference_Smell_minus_NoSmell": difference_scores.mean(),
            "t_value": t_stat,
            "p_value": p_value,
            "Cohens_d": cohens_d
        })

    else:
        ttest_results.append({
            "PRS_Measure": measure_name,
            "NoSmell_Mean": np.nan,
            "NoSmell_SD": np.nan,
            "Smell_Mean": np.nan,
            "Smell_SD": np.nan,
            "Mean_Difference_Smell_minus_NoSmell": np.nan,
            "t_value": np.nan,
            "p_value": np.nan,
            "Cohens_d": np.nan
        })

ttest_results_df = pd.DataFrame(ttest_results)

display(ttest_results_df)

ttest_file = "PRS_PairedTTestResults.xlsx" #Table 5.7

ttest_results_df.to_excel(
    ttest_file,
    index=False
)

print(f"{ttest_file} saved.")

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:423: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return hypotest_fun_in(*args, **kwds)


,PRS_Measure,NoSmell_Mean,NoSmell_SD,Smell_Mean,Smell_SD,Mean_Difference_Smell_minus_NoSmell,t_value,p_value,Cohens_d
0,Total PRS,3.590909,0.064282,5.454545,0.128565,1.863636,4.100000e+01,1.552423e-02,2.899138e+01
1,Being Away,3.666667,0.000000,5.500000,0.235702,1.833333,1.100000e+01,5.771588e-02,7.778175e+00
2,Fascination,3.500000,0.235702,5.500000,0.235702,2.000000,4.503600e+15,1.413580e-16,3.184526e+15
3,Coherence,3.666667,0.000000,5.333333,0.000000,1.666667,inf,0.000000e+00,NaN
4,Scope,3.500000,0.000000,5.500000,0.000000,2.000000,inf,0.000000e+00,NaN


PRS_PairedTTestResults.xlsx saved.


#Download results

In [ ]:
from google.colab import files

files.download("PRS_PrimaryOutcomes.xlsx")            # needed for master dataset
files.download("PRS_DescriptiveStatistics.xlsx")      #Table 5.6
files.download("PRS_NormalityResults.xlsx")           #Table F.3
files.download("PRS_PairedTTestResults.xlsx")         #Table 5.7

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>